In [ ]:
import os
from io import BytesIO

import pandas as pd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient


def load(container_name: str = "baraa") -> dict[str, pd.DataFrame]:
    """Load all CSV files from a container into DataFrames named by filename."""
    # Load env vars for local development and notebooks.
    load_dotenv()

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "AZURE_STORAGE_CONNECTION_STRING is not set in the environment."
        )

    # Connect to Azure Blob Storage and open the target container.
    blob_service_client = BlobServiceClient.from_connection_string(
        connection_string
    )
    container_client = blob_service_client.get_container_client(container_name)

    dataframes: dict[str, pd.DataFrame] = {}

    # Read every CSV found in the container and store it in a reusable dict.
    for blob in container_client.list_blobs():
        if not blob.name.lower().endswith(".csv"):
            continue

        blob_path = blob.name
        variable_name = os.path.splitext(os.path.basename(blob_path))[0]

        print(f"Loading: {blob_path}")
        blob_client = container_client.get_blob_client(blob_path)
        blob_data = blob_client.download_blob().readall()
        dataframe = pd.read_csv(BytesIO(blob_data))

        # Keep both the return dictionary and a global variable for notebook use.
        globals()[variable_name] = dataframe
        dataframes[variable_name] = dataframe

    return dataframes


if __name__ == "__main__":
    try:
        loaded = load()
        print("Loaded files:", ", ".join(sorted(loaded.keys())))

        for name, dataframe in loaded.items():
            print(f"{name} rows: {len(dataframe)}")
    except Exception as exc:
        print(f"Warning: unable to load Azure CSVs: {exc}")


Loading: source_crm/cust_info.csv
Loading: source_crm/prd_info.csv
Loading: source_crm/sales_details.csv
Loading: source_erp/CUST_AZ12.csv
Loading: source_erp/LOC_A101.csv
Loading: source_erp/PX_CAT_G1V2.csv
Loaded files: CUST_AZ12, LOC_A101, PX_CAT_G1V2, cust_info, prd_info, sales_details
cust_info rows: 18494
prd_info rows: 397
sales_details rows: 60398
CUST_AZ12 rows: 18484
LOC_A101 rows: 18484
PX_CAT_G1V2 rows: 37
